# P04 · More encoders (DINOv2, DINOv3 ViT-L web vs satellite) + threshold metrics, development events

Answers three reviewer questions on the **development and secondary** events only (exploratory):

1. **DINOv2 ViT-B/14**, named in the paper as a core encoder, is run under the same probes as DINOv3 and ImageNet.
2. **Does satellite pretraining help?** DINOv3 ViT-L/16 pretrained on satellite images (SAT-493M) against the
   same-size DINOv3 ViT-L/16 pretrained on web images (LVD-1689M). Model size is held fixed so that only the
   pretraining data differs.
3. **Threshold metrics.** AUROC ignores prevalence. For every setting of p03 and p04, F1 at the probe's 0.5
   threshold and the predicted share of damaged buildings against the true share (a direct view of the
   calibration problem).

* All encoders frozen, **outline pooling, native resolution**, same leave-one-event-out probes.
* **Before running:**
  1. Runtime → Change runtime type → **T4 GPU**. Secrets: `GH_TOKEN`, `HF_TOKEN`.
  2. With the Hugging Face account of `HF_TOKEN`, open both model pages and accept the licence (both are gated):
     https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m and
     https://huggingface.co/facebook/dinov3-vitl16-pretrain-sat493m
  3. Needs p01 (building table), p02 (DINOv3 ViT-B features) and p03 (its run folder).
* **Time:** about 90–110 minutes (DINOv2 ~10 min, each ViT-L ~35 min in full precision, bench run ~10 min); about 4 compute units. The ViT-L models run in fp32: in fp16 their activations overflow.
* **Outputs:** features in `02_models/features/<date>_more_encoders/`, the run in this run's folder.

In [ ]:
NOTEBOOK_NAME = "p04_more_encoders_and_thresholds"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
# Install this repo's package (private repo: needs GH_TOKEN in Colab Secrets, read access to damagesar-bench)
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
!git clone -q https://{GH}@github.com/ThinukaLehan/damagesar-bench.git /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
!pip install -q -e /content/damagesar-bench
COMMIT = !git -C /content/damagesar-bench rev-parse --short HEAD
COMMIT = COMMIT[0]
# A running Colab kernel does not see an editable install until a restart: put src/ on the path.
import sys, importlib
if "/content/damagesar-bench/src" not in sys.path:
    sys.path.insert(0, "/content/damagesar-bench/src")
importlib.invalidate_caches()
import damagesar_bench
print("damagesar-bench at commit", COMMIT, "->", damagesar_bench.__file__)

In [ ]:
# GPU, Hugging Face token, inputs from p02 and p03
import glob, os, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as ex:
    raise RuntimeError("HF_TOKEN is missing: add it in Colab -> Secrets (key icon), switch on notebook access, "
                       "then Run all again.") from ex
from huggingface_hub import model_info
for mid in ["facebook/dinov3-vitl16-pretrain-lvd1689m", "facebook/dinov3-vitl16-pretrain-sat493m"]:
    try:
        model_info(mid, token=os.environ["HF_TOKEN"]); print("access OK:", mid)
    except Exception as ex:
        raise RuntimeError(f"No access to {mid}: open https://huggingface.co/{mid} with the HF_TOKEN account, "
                           "accept the licence, then Run all again.") from ex
P02_FEATS = f"{PROJECT}/02_models/features/2026-11-20_dinov3_resolution/features.parquet"
P03_RUN = sorted(glob.glob(f"{PROJECT}/04_outputs/runs/*_p03-controls-and-imagenet-v0/bench_run"))
assert os.path.exists(P02_FEATS), f"missing {P02_FEATS}: run p02 first"
assert P03_RUN, "missing the p03 run folder: run p03 first"
P03_RUN = P03_RUN[-1]
print("GPU:", torch.cuda.get_device_name(0), "| p03 run:", P03_RUN)

In [ ]:
# BRIGHT post-event SAR only (~2 GB): labels come from the building table
import zipfile
from huggingface_hub import hf_hub_download
ROOT = "/content/bright"
post = f"{ROOT}/post-event"
if not (os.path.isdir(post) and len(os.listdir(post)) > 4000):
    p = hf_hub_download("Kullervo/BRIGHT", "post-event.zip", repo_type="dataset", local_dir="/content/zips")
    with zipfile.ZipFile(p) as z:
        z.extractall(ROOT)
print(len(os.listdir(post)), "post-event tiles")

In [ ]:
# Building table from p01: development + secondary events only (confirmatory stay locked)
import datetime, gc
import pandas as pd
from damagesar_bench.config import load_events
events = load_events("/content/damagesar-bench/configs/events.yaml")
B = pd.read_parquet(f"{PROJECT}/01_data/derived/buildings.parquet")
B["damaged"] = B["damaged"].astype("boolean")
use = B[B["role"].isin(["development", "secondary"])].copy()
del B
FEAT_DIR = f"{PROJECT}/02_models/features/{datetime.date.today().isoformat()}_more_encoders"
os.makedirs(FEAT_DIR, exist_ok=True)
print(use[use["excluded_reason"].isna()].groupby("event").size().to_string())

In [ ]:
# Features, outline pooling, native resolution: DINOv2 ViT-B/14, DINOv3 ViT-L/16 (web), DINOv3 ViT-L/16 (satellite)
from damagesar_bench.encoders import get_encoder
from damagesar_bench.encoders.extract import read_bright_tile, write_features
from damagesar_bench.experiments.resolution import extract_at_resolutions, runs_table
FEATS, timing, revisions = {}, [], {}
for name in ["dinov2_vitb14", "dinov3_vitl16", "dinov3_vitl16_sat"]:
    enc = get_encoder(name)
    revisions[name] = enc.weights_revision
    f, runs = extract_at_resolutions(use, lambda t: read_bright_tile(ROOT, t), enc, ["outline"], [None],
                                     f"/content/features/{name}", events, log=print)
    FEATS[name] = str(write_features(f, f"{FEAT_DIR}/{name}.parquet"))
    timing.append(runs_table(runs))
    del f, enc; gc.collect(); torch.cuda.empty_cache()
timing = pd.concat(timing, ignore_index=True)
timing.to_csv(f"{RUN}/timing.csv", index=False)
pd.Series(revisions, name="weights_revision").to_csv(f"{RUN}/weights_revisions.csv")
display(timing); print(revisions)

In [ ]:
# bench run: the three new encoders against DINOv3 ViT-B/16 (p02 features); ImageNet comes from p03
del use; gc.collect()
from IPython.display import Markdown
from damagesar_bench.run import ComparisonSpec, run_benchmark
B16, V2 = "dinov3_vitb16@native", "dinov2_vitb14@native"
L16, SAT = "dinov3_vitl16@native", "dinov3_vitl16_sat@native"
comps = [
    ComparisonSpec((SAT, "outline"), (L16, "outline"), "DINOv3-L satellite vs web"),
    ComparisonSpec((L16, "outline"), (B16, "outline"), "DINOv3 ViT-L vs ViT-B (web)"),
    ComparisonSpec((B16, "outline"), (V2, "outline"), "DINOv3 vs DINOv2 (ViT-B)"),
]
out = run_benchmark([P02_FEATS, FEATS["dinov2_vitb14"], FEATS["dinov3_vitl16"], FEATS["dinov3_vitl16_sat"]],
                    comps, events, f"{RUN}/bench_run", n_boot=2000, only_compared=True)
display(Markdown(open(f"{out}/report.md", encoding="utf-8").read()))
m = pd.read_csv(f"{out}/event_metrics.csv")
pv = m.pivot_table(index=["encoder", "pooling"], columns="event", values="auroc"); pv["mean"] = pv.mean(axis=1)
print(pv.round(3).to_string())

In [ ]:
# Threshold metrics for every setting of p03 and p04: F1 at the probe's 0.5 threshold, and the predicted
# share of damaged buildings against the true share (probes use balanced class weights, so 0.5 is their
# natural threshold; a large gap between predicted and true share is the calibration problem)
from sklearn.metrics import f1_score
S = pd.concat([pd.read_parquet(f"{P03_RUN}/fold_scores.parquet"), pd.read_parquet(f"{out}/fold_scores.parquet")],
              ignore_index=True).drop_duplicates(["encoder", "pooling", "event", "tile_id", "instance_id"])
rows = []
for (enc, pool, ev), g in S.groupby(["encoder", "pooling", "event"]):
    y = g["damaged"].astype(bool).to_numpy(); yhat = g["score"].to_numpy() >= 0.5
    rows.append({"encoder": enc, "pooling": pool, "event": ev, "n": len(g), "true_share": y.mean(),
                 "predicted_share": yhat.mean(), "f1": f1_score(y, yhat, zero_division=0)})
T = pd.DataFrame(rows)
T.to_csv(f"{RUN}/threshold_metrics.csv", index=False)
print(T.pivot_table(index=["encoder", "pooling"], columns="event", values="f1").round(3).to_string())
print(T.pivot_table(index=["encoder", "pooling"], columns="event",
                    values=["true_share", "predicted_share"]).round(3).to_string())

In [ ]:
# File the executed notebook (with outputs) into this run's folder and 03_notebooks/pipeline/
import json
from google.colab import _message
# Ask the browser for the notebook; it can time out (busy tab, large outputs). Results above are already saved.
nb = None
for attempt in range(3):
    try:
        reply = _message.blocking_request("get_ipynb", timeout_sec=120)
        nb = (reply or {}).get("ipynb")
    except Exception as ex:
        print("attempt", attempt + 1, "failed:", ex)
    if nb:
        break
if not nb:
    print("Could not copy the executed notebook automatically (the results above are already saved).")
    print("Use File -> Save a copy in Drive, then move the copy into", RUN)
else:
    stamp = datetime.date.today().isoformat()
    for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
                   f"{PROJECT}/03_notebooks/pipeline/{stamp}_{NOTEBOOK_NAME}.ipynb"):
        with open(target, "w") as fh:
            json.dump(nb, fh)
        print("saved", target)
    print("run folder:", sorted(os.listdir(RUN)))